## APRIORI ALGORITHM IMPLEMENTATION

### Importing Required Libraries

In [5]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

### Loading the Cleaned Transaction Dataset

In [6]:
df = pd.read_csv('Cleaned_Transactions.csv')
df.head()

,BillNo,ItemsList,CustomerID,PresentDate,Country,TotalQuantity
0,537210,"CINAMMON SET OF 9 T-LIGHTS, WHITE HANGING HEAR...",15953,05-12-2010 15:15,UNITED KINGDOM,100
1,557974,"POLYESTER FILLER PAD 40X40CM, CHILDRENS CUTLER...",14541,24-06-2011 10:31,UNITED KINGDOM,100
2,547890,"SET OF 20 KIDS COOKIE CUTTERS, RECYCLED PENCIL...",17227,28-03-2011 10:03,UNITED KINGDOM,100
3,539859,"LARGE WHITE HONEYCOMB PAPER BELL, PINK HONEYC...",17905,22-12-2010 16:23,UNITED KINGDOM,100
4,563698,"ASSORTED COLOUR BIRD ORNAMENT, JUMBO BAG RED R...",14667,18-08-2011 13:46,UNITED KINGDOM,100


### Preparing Transactions for Apriori

In [7]:
transactions_clean = []
for t in df['ItemsList']:
    t = str(t)
    items = t.split(',')  # split by comma
    items = [i.strip().strip('"').strip("'") for i in items if i.strip()]
    transactions_clean.append(items)

### Converting Transactions into a One-Hot Encoded Format

In [8]:
te = TransactionEncoder()
te_ary = te.fit(transactions_clean).transform(transactions_clean)
trans_df = pd.DataFrame(te_ary, columns=te.columns_)
trans_df.head()

,1 HANGER,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,...,ZINC METAL HEART DECORATION,ZINC PLANT POT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Generating Frequent Itemsets with the Apriori Algorithm

In [9]:
freq_itemsets = apriori(trans_df, min_support=0.01, use_colnames=True)
freq_itemsets['itemset_len'] = freq_itemsets['itemsets'].apply(len)
freq_itemsets.to_csv('frequent_itemsets.csv', index=False)
print("Frequent itemsets generated:", len(freq_itemsets))

Frequent itemsets generated: 2330


### Previewing a Sample Transaction

In [10]:
transactions_clean[0]  # first transaction

['CINAMMON SET OF 9 T-LIGHTS',
 'WHITE HANGING HEART T-LIGHT HOLDER',
 'PAINTED METAL PEARS ASSORTED',
 'ASSORTED COLOUR BIRD ORNAMENT',
 'OCEAN SCENT CANDLE IN JEWELLED BOX',
 'VANILLA SCENT CANDLE JEWELLED BOX',
 'WOODEN PICTURE FRAME WHITE FINISH',
 'WOODEN FRAME ANTIQUE WHITE',
 'HAND WARMER UNION JACK',
 'CHRISTMAS LIGHTS 10 SANTAS',
 'LUNCH BAG WOODLAND',
 'LUNCH BAG CARS BLUE',
 'BISCUITS SMALL BOWL LIGHT BLUE',
 'TEA TIME OVEN GLOVE',
 'SMALL MARSHMALLOWS PINK BOWL']

### Checking One-Hot Encoding for a Single Transaction

In [12]:
trans_df.iloc[0][trans_df.iloc[0] == True]

ASSORTED COLOUR BIRD ORNAMENT         True
BISCUITS SMALL BOWL LIGHT BLUE        True
CHRISTMAS LIGHTS 10 SANTAS            True
CINAMMON SET OF 9 T-LIGHTS            True
HAND WARMER UNION JACK                True
LUNCH BAG CARS BLUE                   True
LUNCH BAG WOODLAND                    True
OCEAN SCENT CANDLE IN JEWELLED BOX    True
PAINTED METAL PEARS ASSORTED          True
SMALL MARSHMALLOWS PINK BOWL          True
TEA TIME OVEN GLOVE                   True
VANILLA SCENT CANDLE JEWELLED BOX     True
WHITE HANGING HEART T-LIGHT HOLDER    True
WOODEN FRAME ANTIQUE WHITE            True
WOODEN PICTURE FRAME WHITE FINISH     True
Name: 0, dtype: bool

### Association Rules Generation

In [13]:
rules = association_rules(freq_itemsets, metric="confidence", min_threshold=0.3)
rules['Antecedent'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
rules['Consequent'] = rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))
rules['Antecedent_Length'] = rules['antecedents'].apply(len)
rules['Consequent_Length'] = rules['consequents'].apply(len)
rules = rules[['Antecedent','Consequent','support','confidence','lift','Antecedent_Length','Consequent_Length']]

In [14]:
rules = rules.reset_index(drop=True)
rules.insert(0, 'RuleID', ['R{:04d}'.format(i+1) for i in range(len(rules))])

In [15]:
rules.to_csv('apriori_rules.csv', index=False)
print("Apriori rules generated:", len(rules))
rules.head()

Apriori rules generated: 3664


,RuleID,Antecedent,Consequent,support,confidence,lift,Antecedent_Length,Consequent_Length
0,R0001,FELTCRAFT BUTTERFLY HEARTS,3 STRIPEY MICE FELTCRAFT,0.010916,0.341837,11.592231,1,1
1,R0002,3 STRIPEY MICE FELTCRAFT,FELTCRAFT BUTTERFLY HEARTS,0.010916,0.370166,11.592231,1,1
2,R0003,6 GIFT TAGS 50'S CHRISTMAS,6 GIFT TAGS VINTAGE CHRISTMAS,0.011893,0.496599,21.928938,1,1
3,R0004,6 GIFT TAGS VINTAGE CHRISTMAS,6 GIFT TAGS 50'S CHRISTMAS,0.011893,0.525180,21.928938,1,1
4,R0005,6 GIFT TAGS 50'S CHRISTMAS,PAPER CHAIN KIT 50'S CHRISTMAS,0.010101,0.421769,5.857051,1,1


### Top 5 Association Rules Extraction

In [17]:
import pandas as pd

# Load the rules
rules = pd.read_csv('apriori_rules.csv')

# Sort by Lift descending
top5_rules = rules.sort_values(by='lift', ascending=False).head(5)

# Select only the columns needed for the report
top5_rules = top5_rules[['RuleID', 'Antecedent', 'Consequent', 'support', 'confidence', 'lift']]

# Display
print(top5_rules)

# Optional: save as CSV for reference
top5_rules.to_csv('Top5_Apriori_Rules.csv', index=False)


     RuleID                                         Antecedent  \
3647  R3648              HERB MARKER CHIVES, HERB MARKER THYME   
3618  R3619  HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...   
3626  R3627  HERB MARKER CHIVES, HERB MARKER ROSEMARY, HERB...   
3639  R3640  HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...   
3623  R3624  HERB MARKER BASIL, HERB MARKER CHIVES, HERB MA...   

                                             Consequent  support  confidence  \
3647  HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...  0.01059    0.928571   
3618              HERB MARKER CHIVES, HERB MARKER THYME  0.01059    0.928571   
3626  HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...  0.01059    0.942029   
3639  HERB MARKER CHIVES, HERB MARKER ROSEMARY, HERB...  0.01059    0.902778   
3623  HERB MARKER MINT, HERB MARKER PARSLEY, HERB MA...  0.01059    0.970149   

           lift  
3647  81.422449  
3618  81.422449  
3626  80.307971  
3639  80.307971  
3623  79.397015  


### Generate the rules table (top rules)

In [20]:
rules = rules.sort_values(['lift','confidence','support'], ascending=False)
rules.head(20)

,RuleID,Antecedent,Consequent,support,confidence,lift,Antecedent_Length,Consequent_Length
3618,R3619,"HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...","HERB MARKER CHIVES, HERB MARKER THYME",0.010590,0.928571,81.422449,4,2
3647,R3648,"HERB MARKER CHIVES, HERB MARKER THYME","HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...",0.010590,0.928571,81.422449,2,4
3626,R3627,"HERB MARKER CHIVES, HERB MARKER ROSEMARY, HERB...","HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...",0.010590,0.942029,80.307971,3,3
3639,R3640,"HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...","HERB MARKER CHIVES, HERB MARKER ROSEMARY, HERB...",0.010590,0.902778,80.307971,3,3
3623,R3624,"HERB MARKER BASIL, HERB MARKER CHIVES, HERB MA...","HERB MARKER MINT, HERB MARKER PARSLEY, HERB MA...",0.010590,0.970149,79.397015,3,3
3629,R3630,"HERB MARKER BASIL, HERB MARKER CHIVES, HERB MA...","HERB MARKER MINT, HERB MARKER PARSLEY, HERB MA...",0.010590,0.970149,79.397015,3,3
3636,R3637,"HERB MARKER MINT, HERB MARKER PARSLEY, HERB MA...","HERB MARKER BASIL, HERB MARKER CHIVES, HERB MA...",0.010590,0.866667,79.397015,3,3
3642,R3643,"HERB MARKER MINT, HERB MARKER PARSLEY, HERB MA...","HERB MARKER BASIL, HERB MARKER CHIVES, HERB MA...",0.010590,0.866667,79.397015,3,3
3621,R3622,"HERB MARKER MINT, HERB MARKER PARSLEY, HERB MA...","HERB MARKER BASIL, HERB MARKER CHIVES",0.010590,0.878378,79.286566,4,2
3633,R3634,"HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...","HERB MARKER CHIVES, HERB MARKER PARSLEY, HERB ...",0.010590,0.878378,79.286566,3,3


In [4]:
!pip install mlxtend

  Using cached mlxtend-0.23.4-py3-none-any.whl.metadata (7.3 kB)
Using cached mlxtend-0.23.4-py3-none-any.whl (1.4 MB)


In [23]:
rules.head(20).to_csv("Person1_Insights_Table.csv", index=False)

In [24]:
rules.head(5)

,RuleID,Antecedent,Consequent,support,confidence,lift,Antecedent_Length,Consequent_Length
3618,R3619,"HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...","HERB MARKER CHIVES, HERB MARKER THYME",0.01059,0.928571,81.422449,4,2
3647,R3648,"HERB MARKER CHIVES, HERB MARKER THYME","HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...",0.01059,0.928571,81.422449,2,4
3626,R3627,"HERB MARKER CHIVES, HERB MARKER ROSEMARY, HERB...","HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...",0.01059,0.942029,80.307971,3,3
3639,R3640,"HERB MARKER BASIL, HERB MARKER MINT, HERB MARK...","HERB MARKER CHIVES, HERB MARKER ROSEMARY, HERB...",0.01059,0.902778,80.307971,3,3
3623,R3624,"HERB MARKER BASIL, HERB MARKER CHIVES, HERB MA...","HERB MARKER MINT, HERB MARKER PARSLEY, HERB MA...",0.01059,0.970149,79.397015,3,3


## Rule Insight 1

#### Rule: Herb Marker Basil + Herb Marker Mint + Herb Marker Chives + Herb Marker Rosemary → Herb Marker Chives + Herb Marker Thyme
#### Meaning: Customers who buy many Herb Marker labels like Basil, Mint, Chives, and Rosemary also buy Chives and Thyme labels.
#### Reason: People purchasing herb labels usually want a complete set for their home garden.
#### Action: Display Herb Marker sets together or offer a “Full Herb Marker Pack”.

## Rule Insight 2

#### Rule: Herb Marker Chives + Herb Marker Thyme → Herb Marker Basil + Herb Marker Mint + Herb Marker Rosemary + Herb Marker Chives
#### Meaning: If someone buys Chives and Thyme markers, they often also buy many other Herb Marker labels.
#### Reason: Customers prefer buying multiple herb labels at once for consistency.
#### Action: Suggest a full herb label bundle when someone buys any two markers.

## Rule Insight 3

#### Rule: Herb Marker Chives + Herb Marker Rosemary + Herb Marker Thyme → Herb Marker Basil + Herb Marker Mint + Herb Marker Chives
#### Meaning: Customers buying Chives, Rosemary, and Thyme usually add Basil and Mint labels too.
#### Reason: These are all common herbs used together in kitchens or gardens.
#### Action: Create a “Most Popular Herb Set” including these items.

## Rule Insight 4

#### Rule: Herb Marker Basil + Herb Marker Mint + Herb Marker Rosemary → Herb Marker Chives + Herb Marker Rosemary + Herb Marker Thyme
#### Meaning: People who buy Basil, Mint, and Rosemary markers also add Chives, Rosemary, and Thyme.
#### Reason: They are trying to complete missing herbs from their set.
#### Action: Show customers a complete herb collection to help them pick missing labels.

## Rule Insight 5

#### Rule: Herb Marker Basil + Herb Marker Chives + Herb Marker Mint → Herb Marker Mint + Herb Marker Parsley + Herb Marker Chives
#### Meaning: Buyers of Basil, Chives, and Mint also buy Mint, Parsley, and Chives markers frequently.
#### Reason: They prefer buying multiple herb labels in one shopping trip.
#### Action: Offer a discount or combo for buying 3 or more herb markers together.

## Insight 1: Customers buy herb markers in sets.

#### Many customers purchase multiple herb marker types at the same time (like Basil, Mint, Chives, Thyme).
#### Reason: People who grow herbs usually buy the full set together.
#### Action: Offer a “Herb Marker Combo Pack” or place all herb markers on the same shelf.

## Insight 2: Basil, Mint, and Chives always appear together.

#### In several rules, Basil + Mint often lead to customers also buying Chives or Thyme.
#### Reason: These herbs are commonly grown together in kitchen gardens.
#### Action: Create a 3-item bundle (Basil + Mint + Chives).

## Insight 3: Customers like completing the collection.

#### If a customer buys 2–3 herb markers, they usually buy the remaining ones too.
#### Reason: They want a full set, not individual items.
#### Action: Promote a “Complete Herb Marker Set” with a small discount.

## Insight 4: Cross-selling of herb markers is very strong.

#### Lift values are very high (around 80), which means buying one herb marker strongly increases the chance of buying others.
#### Action: Use “Frequently Bought Together” tags in the store or website.

## Insight 5: Good opportunity for combo deals and bundles.

#### Since all rules involve multiple herb markers, customers will respond well to combo offers.
#### Action: Add Buy 3 Get 1 Free or Set of 4 Markers.